In [ ]:
# ACC102 Track2: Automated Financial Ratio Analysis for Chinese Listed Companies
# Purpose: This script connects to WRDS CSMAR database, cleans financial data,
#          calculates 5 core financial ratios, and generates trend charts for comparison.
# Output: Excel file with ratio results + 5 visualization charts
# Dataset: WRDS CSMAR Financial Master Database (2020-2025 Annual Reports)
import wrds
import pandas as pd
import matplotlib.pyplot as plt
import sys

def financial_analysis(stock_code_map, username):
    print(f"Starting analysis for: {list(stock_code_map.keys())}")
    
    # --------------------------
    # 1. Connect to WRDS Database
    # --------------------------
    # Establish secure connection to WRDS; close immediately after data fetch to avoid idle connections 
    db = wrds.Connection(wrds_username=username)
    
    # --------------------------
    # 2. Prepare & Execute SQL Query
    # --------------------------
    stock_code_list = list(stock_code_map.keys())
    start_date = "2020-01-01"
    end_date = "2025-12-31"
    
    # Select core financial statement columns from CSMAR database 
    selected_columns = "stkcd, accper, typrep, b002000000, a001000000, a001100000, a003000000, b001100000, a002000000, a002100000"
    
    # Dynamic SQL query: adapt to single or multiple stock codes
    if len(stock_code_list) == 1:
        sql_query = f"""
        SELECT {selected_columns}
        FROM csmar.wrds_csmar_financial_master
        WHERE stkcd = '{stock_code_list[0]}'
        AND accper BETWEEN '{start_date}' AND '{end_date}'
        AND typrep = 'A'
        """
    else:
        sql_query = f"""
        SELECT {selected_columns}
        FROM csmar.wrds_csmar_financial_master
        WHERE stkcd IN {tuple(stock_code_list)}
        AND accper BETWEEN '{start_date}' AND '{end_date}'
        AND typrep = 'A'
        """
        
    print("Fetching data from WRDS...")
    raw_data = db.raw_sql(sql_query, date_cols=["accper"])
    db.close() # Close connection immediately after fetching data
    
    # --------------------------
    # 3. Data Cleaning & Preprocessing
    # --------------------------
    # Rename CSMAR's coded column names to readable English for clarity    
    raw_data = raw_data.rename(columns={
        "b002000000": "Net Profit",
        "a001000000": "Total Assets",
        "a001100000": "Total Current Assets",
        "a003000000": "Equity",
        "b001100000": "Revenue",
        "a002000000": "Total Liabilities",
        "a002100000": "Total Current Liabilities"
    })
    
    # Check if all core financial columns exist to avoid runtime errors
    required_cols = ['Net Profit', 'Total Assets', 'Total Current Assets', 'Equity', 'Revenue', 'Total Liabilities', 'Total Current Liabilities']
    
    missing_cols = [col for col in required_cols if col not in raw_data.columns]
    if missing_cols:
        print(f"Warning: Missing columns in data: {missing_cols}")
        return
        
    # Drop rows with missing core data to ensure analysis reliability
    clean_data = raw_data.dropna(subset=required_cols).copy()
    # Keep only December (month=12) reports to use annual data only (consistent across companies)
    clean_data = clean_data[clean_data['accper'].dt.month == 12]
    
    # Convert stock code to string and map to company names for readability
    clean_data['stkcd'] = clean_data['stkcd'].astype(str)
    clean_data['company_name'] = clean_data['stkcd'].replace(stock_code_map)
    clean_data = clean_data.dropna(subset=['company_name'])
    
    # Stop execution if no valid data remains after cleaning
    if clean_data.empty:
        print("No data remaining after cleaning (check company codes).")
        return
        
    # Extract fiscal year from date for time-series analysis
    clean_data['fiscal_year'] = clean_data['accper'].dt.year
    
    # --------------------------
    # 4. Calculate Core Financial Ratios
    # --------------------------
    # ROA (Return on Assets): Measures efficiency of using assets to generate profit
    clean_data['ROA'] = (clean_data['Net Profit'] / clean_data['Total Assets']) * 100
    # ROE (Return on Equity): Measures return to shareholders
    clean_data['ROE'] = (clean_data['Net Profit'] / clean_data['Equity']) * 100 
    # Net Profit Margin: Measures profitability per unit of revenue
    clean_data['Net Profit Margin'] = (clean_data['Net Profit'] / clean_data['Revenue']) * 100
    # Current Ratio: Measures short-term solvency (ability to pay current liabilities)
    clean_data['Current Ratio'] = clean_data['Total Current Assets'] / clean_data['Total Current Liabilities']
    # Total Debt Ratio: Measures long-term financial risk (leverage level)
    clean_data['Total Debt Ratio'] = (clean_data['Total Liabilities'] / clean_data['Total Assets']) * 100 
    
    # --------------------------
    # 5. Prepare & Export Final Results
    # --------------------------
    # Select only relevant columns and sort by company + year for readability
    final_cols = ['company_name', 'fiscal_year', 'ROA', 'ROE', 'Net Profit Margin', 'Current Ratio', 'Total Debt Ratio']
    final_data = clean_data[final_cols].sort_values(['company_name', 'fiscal_year'])
    
    # Export structured results to Excel (no index for cleaner output)
    final_data.to_excel('WRDS_Financial_Analysis_Results.xlsx', index=False)
    print("Data successfully saved to Excel.")
    
    # --------------------------
    # 6. Visualization Setup & Plotting
    # --------------------------
    # Use professional seaborn style for charts
    plt.style.use('seaborn-v0_8-whitegrid')
    
    # Set font size and support both Chinese and English display
    plt.rcParams['font.size'] = 12
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'Arial'] 
    plt.rcParams['axes.unicode_minus'] = False

    def plot_financial_trend(data, ratio_col, title, ylabel, save_name):
        plt.figure(figsize=(10, 5))
        
        # Plot trend for each company present in the data        
        companies_in_data = data['company_name'].unique()
        
        for company in companies_in_data:
            temp = data[data.company_name == company]
            if not temp.empty:
                plt.plot(temp.fiscal_year, temp[ratio_col], marker='o', label=company, linewidth=2)
                
        # Format chart for clarity and professionalism        
        plt.title(title, fontweight='bold')
        plt.xlabel("Fiscal Year")
        plt.ylabel(ylabel)
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_name, bbox_inches='tight', dpi=300)
        plt.show()
        plt.close()
        
    # Generate 5 core financial ratio trend charts
    plot_financial_trend(final_data, "Net Profit Margin", f"Profit Margin ({start_date[:4]}-{end_date[:4]})", "Profit Margin (%)", "Chart_1_Profit_Margin.png")
    plot_financial_trend(final_data, "ROA", f"ROA - Return on Assets ({start_date[:4]}-{end_date[:4]})", "ROA (%)", "Chart_2_ROA.png")
    plot_financial_trend(final_data, "ROE", f"ROE - Return on Equity ({start_date[:4]}-{end_date[:4]})", "ROE (%)", "Chart_3_ROE.png")
    plot_financial_trend(final_data, "Current Ratio", f"Current Ratio ({start_date[:4]}-{end_date[:4]})", "Current Ratio (Times)", "Chart_4_Current_Ratio.png")
    plot_financial_trend(final_data, "Total Debt Ratio", f"Total Debt Ratio ({start_date[:4]}-{end_date[:4]})", "Debt Ratio (%)", "Chart_5_Debt_Ratio.png")

    print("All charts generated successfully.")
    
# --------------------------
# 7. Main Program Entry (User Input)
# --------------------------
if __name__ == "__main__":
    print("Please enter stock code and company name in the format: Stock Code,Company Name")
    print("Example: 000333,Midea")
    print("Press Enter (without typing) to finish input")
    
    stock_code_map = {}
    
    # Loop to accept multiple user inputs    
    while True:
        user_input = input("Enter stock code and company name: ")
        if not user_input:
            break
        
        try:
            code, name = user_input.split(',')
            stock_code_map[code.strip()] = name.strip()
            print(f"Added: {code.strip()} -> {name.strip()}")
        except ValueError:
            print("Invalid format. Please use: Stock Code,Company Name")
            
    # Exit if no valid stock information was entered    
    if not stock_code_map:
        print("No stock information entered, program exiting")
        sys.exit(1)
    
    print(f"\nAnalyzing data for: {list(stock_code_map.values())}")
    
    # --------------------------
    # IMPORTANT: Replace with YOUR OWN WRDS username before running!
    # --------------------------
    username = "yycc" 
    financial_analysis(stock_code_map, username)